# USGS Water Data Demo


This notebook demonstrates how to use the new USGS water data retrieval Python API, [`dataretrieval`](https://doi-usgs.github.io/dataretrieval-python/), to grab the station information needed for the [USGS Hydrologic Toolbox](https://pubs.usgs.gov/tm/04/d03/tm4d3.pdf).


## Useful Links

- [`dataretrieval` documentation](https://doi-usgs.github.io/dataretrieval-python/)
- [`dataretrieval` GitHub repository](https://github.com/DOI-USGS/dataretrieval-python)
- [Introduction to the `waterdata` module example notebook](https://doi-usgs.github.io/dataretrieval-python/examples/index.html)
- [USGS Water Data APIs documentation](https://api.waterdata.usgs.gov/docs/)
- [USGS Monitoring Locations endpoint](https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations?f=html)
- [USGS migration guide for modernized Water Data APIs](https://api.waterdata.usgs.gov/docs/ogcapi/migration/)


In [15]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from dataretrieval import waterdata as wd
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

try:
    from dataretrieval import waterdata as wd
except ImportError as exc:
    raise ImportError("Install `dataretrieval` to run this demo notebook.") from exc

import helpers
from helpers.usgs_waterdata_demo import (
    DEMO_CONFIG,
    NOTEBOOK_MAPPING_ROWS,
    _call_station_discovery_with_filtered_datetime_warning,
    _call_waterdata_with_filtered_datetime_warning,
    _normalize_monitoring_location_id,
    _normalize_station_frame,
    export_timeseries_artifacts,
    format_artifact_messages,
    render_plot_selector,
    render_station_discovery_selector,
)

PROJECT_ROOT = Path(helpers.__file__).resolve().parent.parent

## Sample Data

Demo_Config is dummy data generated to display the type of scenarios we see in the hydrological toolbox.
1. New Project: Bounding Box 
2. Add data: Time series, Parameter code(measurement variable for example ground water 72019), Station
    

In [2]:
DEMO_CONFIG


{'bbox': (-82.859, 31.899, -82.013, 32.947),
 'output_dir': 'outputs/usgs_waterdata_demo',
 'date_range': ('1880-01-01', '2100-01-01'),
 'example_stations': {'discharge': {'station_id': '02225500',
   'parameter_code': '00060'},
  'gw_daily': {'station_id': '324209082430201', 'parameter_code': '72019'},
  'gw_periodic': {'station_id': '410947071344803', 'parameter_code': '72019'}}}

## Legacy to Modern Mapping


In [3]:
pd.DataFrame(NOTEBOOK_MAPPING_ROWS)


,legacy_workflow_step,workflow_type,legacy_intent,modern_demo_entry_point,waterdata_calls,important_filters,expected_output_shape
0,New Project station discovery,discharge,Find surface-water stations in a bounding box,get_usgs_stations_in_bbox,get_monitoring_locations + get_time_series_met...,"site_type_code=ST, parameter_code=00060",DataFrame of discharge-capable stations
1,New Project station discovery,gw_daily,Find groundwater stations with daily series in...,get_usgs_stations_in_bbox,get_monitoring_locations + get_time_series_met...,site_type_code=GW,DataFrame of groundwater daily stations
2,New Project station discovery,gw_periodic,Find groundwater stations with periodic observ...,get_usgs_stations_in_bbox,get_monitoring_locations + get_field_measurements,site_type_code=GW,DataFrame of groundwater periodic stations
3,Adding Data to Project,discharge,Retrieve discharge time series for a selected ...,get_usgs_timeseries,get_daily,"parameter_code=00060, configured date_range",normalized DataFrame + records + files
4,Adding Data to Project,gw_daily,Retrieve groundwater daily time series for a s...,get_usgs_timeseries,get_daily,"configured parameter_code, configured date_range",normalized DataFrame + records + files
5,Adding Data to Project,gw_periodic,Retrieve groundwater periodic measurements for...,get_usgs_timeseries,get_field_measurements,"configured parameter_code, configured date_range",normalized DataFrame + records + files


## Creating New Project

This section mirrors the legacy bounding-box station discovery calls for `discharge`, `gw_daily`, and `gw_periodic`.

The discovered station results are GeoPandas `GeoDataFrame`s, so they carry both tabular station attributes and GIS-style geometry in memory. We are not writing shapefiles here, but you can still inspect `gdf.crs` to confirm the spatial metadata is present.

This section keeps the real `dataretrieval.waterdata` station-discovery calls in the notebook so the demo shows exactly how to use them while matching the old bbox-search theme.

- Old discharge site search theme: `siteType=ST`, `hasDataTypeCd=dv`, `parameterCd=00060`.
- Old gw_daily site search theme: `siteType=GW`, `hasDataTypeCd=dv`.
- Old gw_periodic site search theme: broader bbox station discovery without the daily-value filter.
- `wd.get_monitoring_locations(bbox=bbox, site_type_code="ST" or "GW")` returns monitoring sites inside the bounding box.
- `wd.get_time_series_metadata(bbox=bbox, parameter_code="00060")` narrows those sites to locations with daily time-series metadata.
- `wd.get_field_measurements(bbox=bbox)` finds groundwater stations that have periodic field measurements.
- Use `bbox=(west, south, east, north)` in decimal degrees to mirror the old XML bounding-box inputs.


In [4]:
def get_usgs_stations_in_bbox(bbox, workflow_types):
    """Return workflow-keyed station discovery frames for the legacy new-project step.

    This demo mirrors the old bbox site searches with `waterdata` calls:
    `get_monitoring_locations` finds candidate stations, then
    `get_time_series_metadata` or `get_field_measurements` filters those
    candidates down to the workflow-specific stations we want to preview.
    """
    handlers = {
        "discharge": lambda: _normalize_station_frame(
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_monitoring_locations,
                bbox=bbox,
                site_type_code="ST",
            )[0],
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_time_series_metadata,
                bbox=bbox,
                parameter_code="00060",
            )[0],
            "discharge",
        ),
        "gw_daily": lambda: _normalize_station_frame(
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_monitoring_locations,
                bbox=bbox,
                site_type_code="GW",
            )[0],
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_time_series_metadata,
                bbox=bbox,
            )[0],
            "gw_daily",
        ),
        "gw_periodic": lambda: _normalize_station_frame(
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_monitoring_locations,
                bbox=bbox,
                site_type_code="GW",
            )[0],
            _call_station_discovery_with_filtered_datetime_warning(
                wd.get_field_measurements,
                bbox=bbox,
            )[0],
            "gw_periodic",
        ),
    }

    results = {}
    for workflow_type in workflow_types:
        if workflow_type not in handlers:
            raise ValueError(f"Unsupported workflow type: {workflow_type}")
        results[workflow_type] = handlers[workflow_type]()
    return results


In [5]:
def run_create_new_project_section(bbox, workflow_types):
    """Run the legacy new-project demo section across the requested workflows."""
    return get_usgs_stations_in_bbox(bbox, workflow_types)


In [6]:
create_section_results = run_create_new_project_section(
    DEMO_CONFIG["bbox"],
    ["discharge", "gw_daily", "gw_periodic"],
)

create_preview_columns = [
    "site_no",
    "station_nm",
    "site_tp_cd",
    "dec_lat_va",
    "dec_long_va",
    "state_cd",
    "county_cd",
]

for workflow_type, frame in create_section_results.items():
    print(f"{workflow_type}: {len(frame)} stations")
    print(f"frame type: {type(frame).__name__}")
    print(f"crs: {frame.crs}")
    available_columns = [column for column in create_preview_columns if column in frame.columns]
    preview_columns = available_columns + ["geometry"] if "geometry" in frame.columns else available_columns
    display(frame[preview_columns].head() if preview_columns else frame.head())

discharge: 58 stations
frame type: GeoDataFrame
crs: EPSG:4326


,site_no,station_nm,site_tp_cd,dec_lat_va,dec_long_va,state_cd,county_cd,geometry
0,02200930,"SPRING CREEK NEAR LOUISVILLE, GA",ST,32.922376,-82.313458,13,163,POINT (-82.31346 32.92238)
1,02200950,"OGEECHEE RIVER AT GA 78, NEAR WADLEY, GA",ST,32.869877,-82.319570,13,163,POINT (-82.31957 32.86988)
2,02201110,"NAILS CREEK NEAR BARTOW, GA",ST,32.873767,-82.442628,13,163,POINT (-82.44263 32.87377)
3,02201160,"BOGGY GUT CREEK AT US RT 1 NEAR WADLEY, GA",ST,32.895155,-82.400405,13,163,POINT (-82.4004 32.89515)
4,02201230,"OGEECHEE RIVER AT MIDVILLE, GA",ST,32.814694,-82.235000,13,033,POINT (-82.235 32.81469)


gw_daily: 4 stations
frame type: GeoDataFrame
crs: EPSG:4326


,site_no,station_nm,site_tp_cd,dec_lat_va,dec_long_va,state_cd,county_cd,geometry
0,320226082301101,25Q001,GW,32.040460,-82.501252,13,209,POINT (-82.50125 32.04046)
1,321302082243601,26R001,GW,32.217400,-82.409860,13,279,POINT (-82.40986 32.2174)
2,324209082430201,24V001,GW,32.702662,-82.717081,13,167,POINT (-82.71708 32.70266)
3,325232082131501,28X001,GW,32.875778,-82.220833,13,033,POINT (-82.22083 32.87578)


gw_periodic: 168 stations
frame type: GeoDataFrame
crs: EPSG:4326


,site_no,station_nm,site_tp_cd,dec_lat_va,dec_long_va,state_cd,county_cd,geometry
0,315425082481601,23P002,GW,31.907125,-82.804309,13,271,POINT (-82.80431 31.90712)
1,315433082253401,26P002,GW,31.909358,-82.425963,13,001,POINT (-82.42596 31.90936)
2,315500082063501,29P030,GW,31.916867,-82.109565,13,267,POINT (-82.10956 31.91687)
3,315534082410101,24P007,GW,31.926250,-82.683356,13,271,POINT (-82.68336 31.92625)
4,315545082203201,27P002,GW,31.929083,-82.342071,13,001,POINT (-82.34207 31.92908)


## Adding Data to Project


This section keeps the real `dataretrieval.waterdata` retrieval calls in the notebook so the demo shows the exact request shape for each workflow while matching the old station-pull theme.

- Old discharge add-data theme: daily-values style station pull with `parameterCd=00060`.
- Old gw_daily add-data theme: daily-values style groundwater station pull.
- Old gw_periodic add-data theme: groundwater-level / field-measurement station retrieval rather than a bbox search.
- `wd.get_daily(monitoring_location_id="USGS-...", parameter_code="00060" or another code, time=[start, end])` returns daily values.
- `wd.get_field_measurements(monitoring_location_id="USGS-...", parameter_code="72019", time=[start, end])` returns periodic groundwater field measurements.
- `monitoring_location_id` should use the `USGS-...` prefix expected by `waterdata`.
- `parameter_code` and `time` mirror the old XML inputs for station, variable, and period of record.


In [7]:
def get_usgs_timeseries(
    station_id,
    workflow_type,
    output_dir,
    parameter_code=None,
    date_range=("1880-01-01", "2100-01-01"),
):
    """Return one legacy add-data payload plus exported artifacts for a station.

    `discharge` and `gw_daily` use `wd.get_daily`, while `gw_periodic`
    uses `wd.get_field_measurements` to match the old groundwater-level
    retrieval theme without recreating the raw RDB files.
    """
    monitoring_location_id = _normalize_monitoring_location_id(station_id)

    if workflow_type in {"discharge", "gw_daily"}:
        frame, metadata = _call_waterdata_with_filtered_datetime_warning(
            wd.get_daily,
            monitoring_location_id=monitoring_location_id,
            parameter_code="00060" if workflow_type == "discharge" else parameter_code,
            time=list(date_range),
        )
    elif workflow_type == "gw_periodic":
        frame, metadata = _call_waterdata_with_filtered_datetime_warning(
            wd.get_field_measurements,
            monitoring_location_id=monitoring_location_id,
            parameter_code=parameter_code,
            time=list(date_range),
        )
    else:
        raise ValueError(f"Unsupported workflow type: {workflow_type}")

    metadata_dict = metadata if isinstance(metadata, dict) else vars(metadata).copy()

    frame = frame.reset_index(drop=True).copy()
    frame["station_id"] = station_id
    frame["workflow_type"] = workflow_type

    if frame.empty:
        return {
            "frame": frame,
            "records": [],
            "files": {},
            "metadata": {
                **metadata_dict,
                "warning": f"No records returned for {workflow_type} {station_id}",
            },
        }

    records = frame.to_dict(orient="records")
    file_paths = export_timeseries_artifacts(frame, output_dir, workflow_type, station_id)
    return {
        "frame": frame,
        "records": records,
        "files": file_paths,
        "metadata": metadata_dict,
    }


In [8]:
def run_add_data_to_project_section(example_stations, output_dir, date_range):
    """Run the legacy add-data demo section for each configured workflow type."""
    results = {}
    for workflow_type, example in example_stations.items():
        try:
            payload = get_usgs_timeseries(
                example["station_id"],
                workflow_type,
                output_dir,
                parameter_code=example.get("parameter_code"),
                date_range=date_range,
            )
        except Exception as exc:
            results[workflow_type] = {
                "payload": None,
                "error": str(exc),
                "artifact_messages": [],
            }
            continue

        results[workflow_type] = {
            "payload": payload,
            "error": None,
            "artifact_messages": format_artifact_messages(payload["files"]),
        }
    return results


In [9]:
render_station_discovery_selector(create_section_results)


In [10]:
add_data_results = run_add_data_to_project_section(
    DEMO_CONFIG["example_stations"],
    PROJECT_ROOT / DEMO_CONFIG["output_dir"],
    DEMO_CONFIG["date_range"],
)

for workflow_type, result in add_data_results.items():
    print(workflow_type)
    if result["error"]:
        print(f"Warning: {result['error']}")
        continue

    payload = result["payload"]
    display(payload["frame"].head())
    for message in result["artifact_messages"]:
        print(message)

discharge


,geometry,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id,station_id,workflow_type
0,POINT (-82.1765 32.07714),447f21c4ac494d2a94d69c5079c557c4,USGS-02225500,00060,00003,1903-06-24,1740.0,ft^3/s,Approved,None,2025-03-11 01:35:12.213577+00:00,0de6876f-c8a4-4943-8c1c-994a6d8747d8,02225500,discharge
1,POINT (-82.1765 32.07714),447f21c4ac494d2a94d69c5079c557c4,USGS-02225500,00060,00003,1903-06-25,1830.0,ft^3/s,Approved,None,2025-03-11 01:35:12.213577+00:00,630c037d-28f6-4afd-9694-7e5578b54d04,02225500,discharge
2,POINT (-82.1765 32.07714),447f21c4ac494d2a94d69c5079c557c4,USGS-02225500,00060,00003,1903-06-26,1920.0,ft^3/s,Approved,None,2025-03-11 01:35:12.213577+00:00,261d4374-d726-4cae-b69e-280774ba3cc8,02225500,discharge
3,POINT (-82.1765 32.07714),447f21c4ac494d2a94d69c5079c557c4,USGS-02225500,00060,00003,1903-06-27,2280.0,ft^3/s,Approved,None,2025-03-11 01:35:12.213577+00:00,551ba388-3b70-4b07-b87d-74dd663f2315,02225500,discharge
4,POINT (-82.1765 32.07714),447f21c4ac494d2a94d69c5079c557c4,USGS-02225500,00060,00003,1903-06-28,2760.0,ft^3/s,Approved,None,2025-03-11 01:35:12.213577+00:00,a18a4b2e-6720-40f2-8178-df529e2d007c,02225500,discharge


/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/discharge_02225500.json
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/discharge_02225500.csv
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/discharge_02225500.parquet
gw_daily


,geometry,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id,station_id,workflow_type
0,POINT (-82.71708 32.70266),14b8deb344a04cca9509fe7e92f0127b,USGS-324209082430201,72019,00003,1980-09-25,129.73,ft,Approved,None,2025-03-11 01:25:39.712423+00:00,f6beb0e6-f674-467c-a83f-3e515f61944d,324209082430201,gw_daily
1,POINT (-82.71708 32.70266),14b8deb344a04cca9509fe7e92f0127b,USGS-324209082430201,72019,00003,1980-09-26,129.74,ft,Approved,None,2025-03-11 01:25:39.712423+00:00,3b360be2-b8b0-4315-a1fd-b032ce735f50,324209082430201,gw_daily
2,POINT (-82.71708 32.70266),14b8deb344a04cca9509fe7e92f0127b,USGS-324209082430201,72019,00003,1980-09-27,129.77,ft,Approved,None,2025-03-11 01:25:39.712423+00:00,96ea5520-a3c8-4664-8226-398ee1e5160d,324209082430201,gw_daily
3,POINT (-82.71708 32.70266),14b8deb344a04cca9509fe7e92f0127b,USGS-324209082430201,72019,00003,1980-09-28,129.75,ft,Approved,None,2025-03-11 01:25:39.712423+00:00,347bf4a3-e8a8-4e8c-af76-77e0217a2cd1,324209082430201,gw_daily
4,POINT (-82.71708 32.70266),14b8deb344a04cca9509fe7e92f0127b,USGS-324209082430201,72019,00003,1980-09-29,129.67,ft,Approved,None,2025-03-11 01:25:39.712423+00:00,578c7c83-6f91-4b80-a79d-7d5a1468fa47,324209082430201,gw_daily


/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_daily_324209082430201.json
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_daily_324209082430201.csv
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_daily_324209082430201.parquet
gw_periodic


,geometry,field_visit_id,parameter_code,monitoring_location_id,observing_procedure_code,observing_procedure,value,unit_of_measure,time,qualifier,vertical_datum,approval_status,measuring_agency,last_modified,control_condition,measurement_rated,field_measurement_id,station_id,workflow_type
0,POINT (-71.5795 41.16316),d551dca7-ba61-4902-9042-b8d7c5d510f0,72019,USGS-410947071344803,S,"GW level, steel tape",11.52,ft,1989-10-25 12:00:00+00:00,[Static],Local Assumed Datum,Approved,USGS,2026-02-15 02:34:49.761079+00:00,None,None,448b35b8-39d7-4a21-8252-5363c55d345a,410947071344803,gw_periodic
1,POINT (-71.5795 41.16316),4df76e66-da07-4fb2-98c2-55247a16c87f,72019,USGS-410947071344803,S,"GW level, steel tape",11.99,ft,1990-08-27 20:43:00+00:00,[Static],Local Assumed Datum,Approved,None,2026-02-15 02:34:49.761079+00:00,None,None,14c4b8f1-e679-4511-972d-6123629f775a,410947071344803,gw_periodic
2,POINT (-71.5795 41.16316),8b09326b-c126-4b15-be1f-f349e21b93b5,72019,USGS-410947071344803,S,"GW level, steel tape",12.52,ft,1990-09-21 12:53:00+00:00,[Static],Local Assumed Datum,Approved,None,2026-02-15 02:34:49.761079+00:00,None,None,4963b81e-e610-47cb-b2de-24c7ed46109b,410947071344803,gw_periodic
3,POINT (-71.5795 41.16316),6114810c-7f09-459e-98b5-9f0f6cc30598,72019,USGS-410947071344803,S,"GW level, steel tape",11.25,ft,1991-06-13 20:30:00+00:00,[Static],Local Assumed Datum,Approved,None,2026-02-15 02:34:49.761079+00:00,None,None,2a6c8f5a-127e-43c7-a59e-e3fa13f8961d,410947071344803,gw_periodic
4,POINT (-71.5795 41.16316),548da81c-b404-482b-9414-a31d5d6c44ba,72019,USGS-410947071344803,S,"GW level, steel tape",12.75,ft,1991-08-30 12:00:00+00:00,[Static],Local Assumed Datum,Approved,None,2026-02-15 02:34:49.761079+00:00,None,None,84f54709-5697-48b4-b620-86cfb2916741,410947071344803,gw_periodic


/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_periodic_410947071344803.json
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_periodic_410947071344803.csv
/Users/lludden/Documents/GitHub/WaterData ToolBox Demo/outputs/usgs_waterdata_demo/gw_periodic_410947071344803.parquet


## Plot Generated Outputs


This section plots the generated station-based outputs created above, keeping the add-data widget focused on the daily-values versus groundwater-level retrieval differences.


In [11]:
# Plot frame preparation lives in helpers.usgs_waterdata_demo._prepare_plot_frame.


In [12]:
# Plot rendering lives in helpers.usgs_waterdata_demo.plot_generated_output.


In [13]:
# Widget rendering lives in helpers.usgs_waterdata_demo.render_plot_selector.


In [14]:
render_plot_selector(add_data_results)
